# Day 15: Local Qdrant Instance and Vector Collections

## Core Theory (Just-in-Time)

Vector databases like Qdrant are designed to store, manage, and search high-dimensional vectors. While traditional databases index scalar data, vector databases index embeddings (arrays of floats) using algorithms like HNSW (Hierarchical Navigable Small World) for fast approximate nearest neighbor (ANN) search.

**Why Qdrant?**
Qdrant is written in Rust, offering high performance and memory safety. It supports payload filtering alongside vector search, making it ideal for Retrieval-Augmented Generation (RAG) applications.

**Why Docker for Local Development?**
Running Qdrant via Docker ensures environment consistency between your local machine and production. You don't have to worry about missing dependencies or conflicting database versions on your local OS.

**Vector Dimensions:**
When creating a collection, you must specify the vector's `size` (dimensions) and the `distance` metric (e.g., Cosine, Dot, or Euclidean). The size must exactly match the output dimension of the embedding model you plan to use (e.g., 1536 for OpenAI's `text-embedding-ada-002`).

### Instructions to Run Qdrant locally
Before executing the code, spin up a local instance of Qdrant in your terminal using Docker:
```bash
docker run -d -p 6333:6333 -p 6334:6334 qdrant/qdrant
```
- Port `6333` is for the HTTP API.
- Port `6334` is for the gRPC API (faster, often used in production).

*(Note: The code below uses `location=":memory:"` as a fallback so this notebook can execute natively without requiring the Docker daemon to be running, but in production, you connect to `localhost:6333`.)*

In [1]:
from qdrant_client import QdrantClient
from qdrant_client.http import models
from typing import List, Dict, Any

def initialize_qdrant_client(in_memory: bool = True) -> QdrantClient:
    """
    Initializes a connection to Qdrant.
    
    Args:
        in_memory: If True, uses local memory (for testing). 
                   If False, connects to a local Docker instance.
                   
    Returns:
        An instance of QdrantClient.
    """
    if in_memory:
        return QdrantClient(location=":memory:")
    
    return QdrantClient(url="http://localhost:6333")

def create_vector_collection(
    client: QdrantClient, 
    collection_name: str, 
    vector_size: int
) -> bool:
    """
    Creates a new collection in Qdrant if it doesn't already exist.
    
    Args:
        client: The instantiated QdrantClient.
        collection_name: The name of the collection to create.
        vector_size: The dimensionality of the vectors to be stored.
        
    Returns:
        True if the collection exists or was successfully created, False otherwise.
    """
    # Check if the collection already exists
    if client.collection_exists(collection_name=collection_name):
        print(f"Collection '{collection_name}' already exists.")
        return True
        
    # Create the collection
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=vector_size,
            distance=models.Distance.COSINE
        )
    )
    print(f"Collection '{collection_name}' created successfully with dimension {vector_size}.")
    return client.collection_exists(collection_name=collection_name)

# Implementation execution
if __name__ == "__main__":
    qdrant_client = initialize_qdrant_client(in_memory=True)
    
    COLLECTION_NAME = "day_15_test_collection"
    VECTOR_DIMENSION = 384 # Typical for small models like all-MiniLM-L6-v2
    
    success = create_vector_collection(
        client=qdrant_client,
        collection_name=COLLECTION_NAME,
        vector_size=VECTOR_DIMENSION
    )
    
    assert success is True

Collection 'day_15_test_collection' created successfully with dimension 384.


## Common Pitfalls

1. **Dimension Mismatch:** The most frequent error is trying to insert a vector of size 1536 into a collection initialized with size 384. This throws a hard error in Qdrant.
2. **Wrong Distance Metric:** Using Euclidean distance when the embeddings were normalized for Cosine similarity will yield poor retrieval results. Always check your embedding model's documentation for the recommended distance metric.
3. **Missing Indexes in Production:** While small datasets work fine out of the box, production systems require payload indexes on frequently filtered metadata fields to avoid full collection scans.
4. **Ignoring gRPC:** Using the REST API (port 6333) for high-throughput production workloads instead of the faster gRPC interface (port 6334).

## Practical Lab / Homework

**Task:**
1. Use the initialized `QdrantClient` and collection (`day_15_test_collection`).
2. Upsert three mock vectors into the collection. Each vector must match the `VECTOR_DIMENSION` (384) configured earlier.
3. Include a payload (metadata) for each vector containing at least a `document_id` and a `text_chunk`.
4. Perform a simple nearest-neighbor search using a mock query vector to retrieve the top 2 results.

**Constraint:** 
Write clean, production-ready Python code with strict type hints and docstrings. Do not use pseudo-code.

In [2]:
from qdrant_client import QdrantClient
import random
from qdrant_client.http.models import PointStruct, ScoredPoint
from typing import List

def upsert_mock_vectors(
    client: QdrantClient, 
    collection_name: str, 
    dimension: int
) -> None:
    """
    Upserts a set of mock vectors with payloads into the specified collection.
    
    Args:
        client: The instantiated QdrantClient.
        collection_name: The target collection name.
        dimension: The size of the vectors.
    """
    # Generating 3 mock vectors
    points = [
        PointStruct(
            id=1,
            vector=[random.random() for _ in range(dimension)],
            payload={"document_id": "doc_001", "text_chunk": "Artificial Intelligence is transforming software."}
        ),
        PointStruct(
            id=2,
            vector=[random.random() for _ in range(dimension)],
            payload={"document_id": "doc_002", "text_chunk": "Vector databases are essential for RAG."}
        ),
        PointStruct(
            id=3,
            vector=[random.random() for _ in range(dimension)],
            payload={"document_id": "doc_003", "text_chunk": "LangChain provides abstractions for LLMs."}
        )
    ]
    
    operation_info = client.upsert(
        collection_name=collection_name,
        points=points
    )
    print(f"Upsert operation status: {operation_info.status}")

def search_similar_vectors(
    client: QdrantClient, 
    collection_name: str, 
    query_vector: List[float], 
    limit: int = 2
) -> List[ScoredPoint]:
    """
    Searches the collection for the nearest neighbors to a query vector.
    
    Args:
        client: The instantiated QdrantClient.
        collection_name: The collection to search in.
        query_vector: The vector to compare against.
        limit: The maximum number of results to return.
        
    Returns:
        A list of ScoredPoint objects representing the nearest neighbors.
    """
    search_result = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=limit
    )
    return search_result.points

# Lab Execution
if __name__ == "__main__":
    # We reuse the client and collection details from the previous cell
    upsert_mock_vectors(
        client=qdrant_client,
        collection_name=COLLECTION_NAME,
        dimension=VECTOR_DIMENSION
    )
    
    # Create a random query vector matching the dimension
    mock_query = [random.random() for _ in range(VECTOR_DIMENSION)]
    
    results = search_similar_vectors(
        client=qdrant_client,
        collection_name=COLLECTION_NAME,
        query_vector=mock_query,
        limit=2
    )
    
    print("\nSearch Results:")
    for result in results:
        print(f"ID: {result.id}, Score: {result.score:.4f}, Payload: {result.payload}")


Upsert operation status: completed

Search Results:
ID: 2, Score: 0.7532, Payload: {'document_id': 'doc_002', 'text_chunk': 'Vector databases are essential for RAG.'}
ID: 1, Score: 0.7331, Payload: {'document_id': 'doc_001', 'text_chunk': 'Artificial Intelligence is transforming software.'}
